# CONUS Spatial Data Harmonization

Aligns all raw CONUS satellite and model data to the NLDAS 0.125° grid and saves to `data/processed/conus/`. Run this notebook once after all download scripts (16–19, 06) complete.

**CONUS NLDAS grid**: 464 columns × 224 rows, 0.125° spacing, EPSG:4326

**Pipeline**: downloads → *this notebook* → `01_human_et_conus.ipynb`

In [ ]:
import sys
import os
import shutil
import warnings
from pathlib import Path

import numpy as np
import xarray as xr
import rioxarray as rxr
import rasterio
from rasterio.transform import from_origin
from rasterio.enums import Resampling
from rasterio.warp import reproject, calculate_default_transform
import geopandas as gpd
import pandas as pd

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

# ── Project root ──────────────────────────────────────────────────────────────
_root_env    = os.environ.get('SIF_ROOT')
project_root = Path(_root_env) if _root_env else Path('../../..').resolve()

# ── Raw input directories ─────────────────────────────────────────────────────
raw        = project_root / 'data' / 'raw'
cdl_dir    = raw / 'cdl'
openet_dir = raw / 'openet'
nldas_dir  = raw / 'nldas'
ndvi_dir   = raw / 'ndvi'
sif_dir    = raw / 'sif'
usdm_dir   = raw / 'drought_usdm'
gmet_dir   = raw / 'drought_gridmet'

# ── Processed output ──────────────────────────────────────────────────────────
proc_dir = project_root / 'data' / 'processed' / 'conus'
proc_dir.mkdir(parents=True, exist_ok=True)
for sub in ['cdl', 'openet', 'nldas', 'ndvi', 'sif', 'drought_usdm', 'drought_gridmet']:
    (proc_dir / sub).mkdir(exist_ok=True)

# ── Full CONUS NLDAS download grid (464×224) ──────────────────────────────────
# This is the grid all raw files were downloaded onto.
FULL_WIDTH, FULL_HEIGHT = 464, 224

# ── Clipped analysis grid (OpenET-aligned, 325×189) ──────────────────────────
# OpenET v2.0 ensemble covers lon -124.75→-84.125°W, lat 25.75→49.375°N.
# Clipping to this extent removes Canada, Mexico, and eastern no-data region.
# Maps to cols 2:327, rows 29:218 of the full 464×224 grid.
CLIP_COL = slice(2, 327)   # 325 columns
CLIP_ROW = slice(29, 218)  # 189 rows

CONUS_CRS          = 'EPSG:4326'
CONUS_WIDTH        = 325
CONUS_HEIGHT       = 189
CONUS_LON          = np.arange(-124.6875, -84.0625, 0.125)   # 325 cell centers
CONUS_LAT          = np.arange(  49.3125,  25.6875, -0.125)  # 189 cell centers
NLDAS_AFFINE       = from_origin(-124.75, 49.375, 0.125, 0.125)
CONUS_TRANSFORM    = [0.125, 0, -124.75, 0, -0.125, 49.375]

YEARS = list(range(2015, 2025))


def crop_raster_to_openet(src_path, dst_path, nodata=None):
    """
    Crop a full-CONUS 224×464 GeoTIFF to the 189×325 OpenET-aligned subgrid
    and write to dst_path with corrected metadata. Handles all band counts.
    """
    with rasterio.open(src_path) as src:
        meta = src.meta.copy()
        data = src.read()[:, CLIP_ROW, CLIP_COL]
        nd   = nodata if nodata is not None else src.nodata
    meta.update({
        'height'   : CONUS_HEIGHT,
        'width'    : CONUS_WIDTH,
        'transform': NLDAS_AFFINE,
        'crs'      : CONUS_CRS,
        'nodata'   : nd,
    })
    with rasterio.open(dst_path, 'w', **meta) as dst:
        dst.write(data)


def needs_reprocess(dst_path):
    """Return True if dst does not exist or has the wrong (pre-clip) dimensions."""
    if not dst_path.exists():
        return True
    try:
        with rasterio.open(dst_path) as src:
            return src.height != CONUS_HEIGHT or src.width != CONUS_WIDTH
    except Exception:
        return True


print('Project root:', project_root)
print('Output dir  :', proc_dir)
print(f'Analysis grid: {CONUS_WIDTH}×{CONUS_HEIGHT} | '
      f'LON {CONUS_LON[0]:.4f}→{CONUS_LON[-1]:.4f} | '
      f'LAT {CONUS_LAT[0]:.4f}→{CONUS_LAT[-1]:.4f}')

Project root: /home/pielab-sandbox-jcoldiron/SIF-Analysis
Output dir  : /home/pielab-sandbox-jcoldiron/SIF-Analysis/data/processed/conus
Analysis grid: 325×189 | LON -124.6875→-84.1875 | LAT 49.3125→25.8125


## 1. CDL — Crop Type Fractions

Reprojects GEE-exported CDL fraction rasters (from `16_download_cdl_conus.py`) to the NLDAS CONUS grid using bilinear resampling (appropriate since values are already fractions, not categorical codes). Outputs one multi-band GeoTIFF per year with bands: `cropland_frac`, `corn_frac`, `soy_frac`, `wheat_frac`, `cotton_frac`, `pasture_frac`.

In [ ]:
import rasterio
from rasterio.warp import reproject, Resampling as WarpResampling
import numpy as np

CDL_BAND_NAMES = ['cropland_frac', 'corn_frac', 'soy_frac', 'wheat_frac', 'cotton_frac', 'pasture_frac']

processed_cdl = []
missing_cdl   = []

for year in YEARS:
    src_path = cdl_dir / ('CDL_CONUS_' + str(year) + '.tif')
    dst_path = proc_dir / 'cdl' / ('CDL_CONUS_' + str(year) + '.tif')

    if dst_path.exists() and not needs_reprocess(dst_path):
        print('CDL', year, '— already processed, skipping.')
        processed_cdl.append(year)
        continue

    if dst_path.exists():
        dst_path.unlink()
        print('CDL', year, '— stale file removed, reprocessing...')

    if not src_path.exists():
        print('WARNING: CDL source not found:', src_path)
        missing_cdl.append(year)
        continue

    print('CDL', year, '— reprojecting to clipped grid...')
    with rasterio.open(src_path) as src:
        n_bands = src.count
        src_crs = src.crs

        dst_data = np.zeros((n_bands, CONUS_HEIGHT, CONUS_WIDTH), dtype=np.float32)

        reproject(
            source=rasterio.band(src, list(range(1, n_bands + 1))),
            destination=dst_data,
            src_transform=src.transform,
            src_crs=src_crs,
            dst_transform=NLDAS_AFFINE,
            dst_crs=CONUS_CRS,
            resampling=WarpResampling.bilinear,
            dst_nodata=np.nan,
        )

        out_meta = {
            'driver'   : 'GTiff',
            'dtype'    : 'float32',
            'width'    : CONUS_WIDTH,
            'height'   : CONUS_HEIGHT,
            'count'    : n_bands,
            'crs'      : CONUS_CRS,
            'transform': NLDAS_AFFINE,
            'nodata'   : np.nan,
            'compress' : 'lzw',
        }

        with rasterio.open(dst_path, 'w', **out_meta) as dst:
            for b in range(n_bands):
                dst.write(dst_data[b], b + 1)
            if n_bands == len(CDL_BAND_NAMES):
                for i, name in enumerate(CDL_BAND_NAMES, 1):
                    dst.update_tags(i, name=name)

    processed_cdl.append(year)
    print('  saved:', dst_path.name, f'({CONUS_WIDTH}×{CONUS_HEIGHT})')

latest_cdl = sorted(processed_cdl)[-1] if processed_cdl else None
if latest_cdl:
    qc_path = proc_dir / 'cdl' / ('CDL_CONUS_' + str(latest_cdl) + '.tif')
    with rasterio.open(qc_path) as src:
        print(f'\nQC — CDL {latest_cdl} | {src.count} bands | {src.width}×{src.height} | CRS: {src.crs.to_epsg()}')
        for b in range(1, src.count + 1):
            band_data = src.read(b, masked=True)
            label = CDL_BAND_NAMES[b - 1] if b - 1 < len(CDL_BAND_NAMES) else 'band_' + str(b)
            print(f'  band {b} ({label}): min={float(band_data.min()):.4f} max={float(band_data.max()):.4f}')

print(f'\nCDL summary: processed {len(processed_cdl)} years | missing {len(missing_cdl)} years')

CDL 2015 — stale file removed, reprocessing...
CDL 2015 — reprojecting to clipped grid...
  saved: CDL_CONUS_2015.tif (325×189)
CDL 2016 — stale file removed, reprocessing...
CDL 2016 — reprojecting to clipped grid...
  saved: CDL_CONUS_2016.tif (325×189)
CDL 2017 — stale file removed, reprocessing...
CDL 2017 — reprojecting to clipped grid...
  saved: CDL_CONUS_2017.tif (325×189)
CDL 2018 — stale file removed, reprocessing...
CDL 2018 — reprojecting to clipped grid...
  saved: CDL_CONUS_2018.tif (325×189)
CDL 2019 — stale file removed, reprocessing...
CDL 2019 — reprojecting to clipped grid...
  saved: CDL_CONUS_2019.tif (325×189)
CDL 2020 — stale file removed, reprocessing...
CDL 2020 — reprojecting to clipped grid...
  saved: CDL_CONUS_2020.tif (325×189)
CDL 2021 — stale file removed, reprocessing...
CDL 2021 — reprojecting to clipped grid...
  saved: CDL_CONUS_2021.tif (325×189)
CDL 2022 — stale file removed, reprocessing...
CDL 2022 — reprojecting to clipped grid...
  saved: CDL_C

## 2. OpenET — Monthly Ensemble ET

OpenET CONUS files are already on the NLDAS grid (downloaded by `17_download_openet_conus.py`). This step verifies alignment and copies/links them to the processed directory.

**Unit**: mm/month

In [ ]:
openet_files = sorted(openet_dir.glob('OpenET_CONUS_*.tif'))

if not openet_files:
    print('WARNING: No OpenET CONUS files found in', openet_dir)
else:
    print(f'Found {len(openet_files)} OpenET CONUS files')

valid_openet   = []
invalid_openet = []

for fpath in openet_files:
    dst = proc_dir / 'openet' / fpath.name

    if dst.exists() and not needs_reprocess(dst):
        valid_openet.append(fpath.name)
        continue

    if dst.exists():
        dst.unlink()

    # Validate source is the expected full-CONUS grid
    try:
        with rasterio.open(fpath) as src:
            if src.height != FULL_HEIGHT or src.width != FULL_WIDTH:
                print(f'WARNING: unexpected source shape in {fpath.name}: {src.height}×{src.width}')
                invalid_openet.append(fpath.name)
                continue
    except Exception as e:
        print(f'WARNING: Could not open {fpath.name}: {e}')
        invalid_openet.append(fpath.name)
        continue

    crop_raster_to_openet(fpath, dst, nodata=float('nan'))
    valid_openet.append(fpath.name)

print(f'Valid and cropped to {CONUS_WIDTH}×{CONUS_HEIGHT}: {len(valid_openet)} | Invalid: {len(invalid_openet)}')

# Sample QC
sample_july22 = proc_dir / 'openet' / 'OpenET_CONUS_202207.tif'
if sample_july22.exists():
    with rasterio.open(sample_july22) as src:
        data = src.read(1, masked=True)
        print(f'\nSample — OpenET July 2022 ({src.width}×{src.height}, CRS: {src.crs.to_epsg()}):')
        print(f'  mean={float(data.mean()):.2f}  min={float(data.min()):.2f}  max={float(data.max()):.2f} mm/month')
        print(f'  valid pixels: {(~np.isnan(src.read(1).astype(float))).sum()} of {src.width*src.height}')

# Missing months check
expected_months = [str(y) + str(m).zfill(2) for y in YEARS for m in range(1, 13)]
available_stems = {f.replace('OpenET_CONUS_', '').replace('.tif', '') for f in valid_openet}
missing_months  = [ym for ym in expected_months if ym not in available_stems]
if missing_months:
    print(f'\nWARNING: Missing OpenET months ({len(missing_months)}): {missing_months[:12]}')
else:
    print('All 120 expected OpenET months present.')

Found 120 OpenET CONUS files
Valid and cropped to 325×189: 120 | Invalid: 0

Sample — OpenET July 2022 (325×189, CRS: 4326):
  mean=105.14  min=1.00  max=294.00 mm/month
  valid pixels: 41393 of 61425
All 120 expected OpenET months present.


## 3. NLDAS Noah — Actual ET

Loads the full-CONUS NLDAS Evap files (saved by `06_download_nldas.py`) and resamples to the NLDAS CONUS grid. NLDAS is already at 0.125° so this is mainly a coordinate-alignment pass.

**Unit**: mm/month (kg/m²)

In [ ]:
nldas_files = sorted(nldas_dir.glob('NLDAS_NOAH0125_M.A*.nc'))

if not nldas_files:
    print('WARNING: No NLDAS NetCDF files found in', nldas_dir)
else:
    print(f'Found {len(nldas_files)} NLDAS files')

processed_nldas = []
missing_nldas   = []
sample_stats    = None

for fpath in nldas_files:
    try:
        yyyymm = fpath.name.split('.A')[1][:6]
        year   = int(yyyymm[:4])
    except (IndexError, ValueError):
        print(f'WARNING: Could not parse date from {fpath.name} — skipping')
        continue

    if year not in YEARS:
        continue

    dst_path = proc_dir / 'nldas' / ('NLDAS_Evap_' + yyyymm + '.nc')

    if dst_path.exists() and not needs_reprocess(dst_path):
        processed_nldas.append(yyyymm)
        continue

    if dst_path.exists():
        dst_path.unlink()

    try:
        ds = xr.open_dataset(fpath)
    except Exception as e:
        print(f'WARNING: Could not open {fpath.name}: {e}')
        missing_nldas.append(yyyymm)
        continue

    evap_var = None
    for candidate in ['Evap', 'evap', 'EVAP', 'ET', 'aevap_f']:
        if candidate in ds.data_vars:
            evap_var = candidate
            break
    if evap_var is None:
        print(f'WARNING: No Evap variable in {fpath.name} | vars: {list(ds.data_vars)}')
        ds.close()
        missing_nldas.append(yyyymm)
        continue

    da = ds[evap_var].squeeze(drop=True)

    lat_dim = lon_dim = None
    for dim in da.dims:
        if dim.lower() in ('lat', 'latitude', 'y'):  lat_dim = dim
        if dim.lower() in ('lon', 'longitude', 'x'): lon_dim = dim

    if lat_dim is None or lon_dim is None:
        print(f'WARNING: Could not identify lat/lon dims in {fpath.name} | dims: {list(da.dims)}')
        ds.close()
        missing_nldas.append(yyyymm)
        continue

    rename_map = {}
    if lat_dim != 'lat': rename_map[lat_dim] = 'lat'
    if lon_dim != 'lon': rename_map[lon_dim] = 'lon'
    if rename_map:
        da = da.rename(rename_map)

    # Reindex to the clipped CONUS_LAT/LON (removes Canada, Mexico, eastern gap)
    da = da.reindex(lat=CONUS_LAT, lon=CONUS_LON, method='nearest', tolerance=0.07)

    ds_out = da.to_dataset(name='Evap')
    ds_out['Evap'].attrs.update({
        'units'    : 'kg/m^2 (mm/month)',
        'long_name': 'Total Evapotranspiration',
        'source'   : 'NLDAS Noah Land Surface Model',
        'yyyymm'   : yyyymm,
    })
    ds_out.attrs['crs'] = CONUS_CRS
    ds_out.to_netcdf(dst_path)
    ds.close()

    if sample_stats is None:
        arr   = da.values
        valid = arr[np.isfinite(arr)]
        sample_stats = (yyyymm, float(valid.mean()), float(valid.min()), float(valid.max()))

    processed_nldas.append(yyyymm)

print(f'\nNLDAS summary: processed {len(processed_nldas)} | missing {len(missing_nldas)}')
if sample_stats:
    print(f'Sample ({sample_stats[0]}): mean={sample_stats[1]:.2f}  min={sample_stats[2]:.2f}  max={sample_stats[3]:.2f} mm/month')

# QC: verify dimensions of a saved file
qc_nldas = proc_dir / 'nldas' / 'NLDAS_Evap_202207.nc'
if qc_nldas.exists():
    ds_chk = xr.open_dataset(qc_nldas)
    lat_n = len(ds_chk.lat) if 'lat' in ds_chk.coords else '?'
    lon_n = len(ds_chk.lon) if 'lon' in ds_chk.coords else '?'
    print(f'QC dims: {lon_n} lon × {lat_n} lat  (expected {CONUS_WIDTH}×{CONUS_HEIGHT})')
    ds_chk.close()

if missing_nldas:
    print(f'Missing months: {missing_nldas}')

Found 120 NLDAS files

NLDAS summary: processed 120 | missing 0
Sample (201501): mean=8.48  min=-5.06  max=47.55 mm/month
QC dims: 325 lon × 189 lat  (expected 325×189)


## 4. NDVI — MODIS MOD13A3

NDVI CONUS files are already on the NLDAS grid (downloaded by `18_download_ndvi_conus.py`). Verifies and copies to processed directory.

**Unit**: dimensionless (−1 to 1)

In [ ]:
ndvi_files = sorted(ndvi_dir.glob('NDVI_CONUS_*.tif'))

if not ndvi_files:
    print('WARNING: No NDVI CONUS files found in', ndvi_dir)
else:
    print(f'Found {len(ndvi_files)} NDVI CONUS files')

valid_ndvi   = []
invalid_ndvi = []

for fpath in ndvi_files:
    dst = proc_dir / 'ndvi' / fpath.name

    if dst.exists() and not needs_reprocess(dst):
        valid_ndvi.append(fpath.name)
        continue

    if dst.exists():
        dst.unlink()

    try:
        with rasterio.open(fpath) as src:
            if src.height != FULL_HEIGHT or src.width != FULL_WIDTH:
                print(f'WARNING: unexpected source shape in {fpath.name}: {src.height}×{src.width}')
                invalid_ndvi.append(fpath.name)
                continue
    except Exception as e:
        print(f'WARNING: Could not open {fpath.name}: {e}')
        invalid_ndvi.append(fpath.name)
        continue

    crop_raster_to_openet(fpath, dst)
    valid_ndvi.append(fpath.name)

print(f'Valid and cropped to {CONUS_WIDTH}×{CONUS_HEIGHT}: {len(valid_ndvi)} | Invalid: {len(invalid_ndvi)}')

# Sample QC
sample_ndvi = proc_dir / 'ndvi' / 'NDVI_CONUS_202207.tif'
if sample_ndvi.exists():
    with rasterio.open(sample_ndvi) as src:
        data = src.read(1, masked=True)
        print(f'\nSample — NDVI July 2022 ({src.width}×{src.height}, CRS: {src.crs.to_epsg()}):')
        print(f'  mean={float(data.mean()):.4f}  min={float(data.min()):.4f}  max={float(data.max()):.4f}')

expected_months = [str(y) + str(m).zfill(2) for y in YEARS for m in range(1, 13)]
available_stems = {f.replace('NDVI_CONUS_', '').replace('.tif', '') for f in valid_ndvi}
missing_months  = [ym for ym in expected_months if ym not in available_stems]
if missing_months:
    print(f'\nWARNING: Missing NDVI months ({len(missing_months)}): {missing_months[:12]}')
else:
    print('All 120 expected NDVI months present.')

Found 120 NDVI CONUS files
Valid and cropped to 325×189: 120 | Invalid: 0

Sample — NDVI July 2022 (325×189, CRS: 4326):
  mean=0.4955  min=-0.1884  max=1.0000
All 120 expected NDVI months present.


## 5. SIF — OCO-2 v11r

Subsets OCO-2 SIF half-monthly NetCDF files to the CONUS extent and bins them onto the NLDAS 0.125° grid by computing the mean SIF within each 0.125° cell. OCO-2 SIF (0.05° native) → 0.125° NLDAS.

**Variable**: `sif_ann` (Annual/Daily SIF, mW/m²/sr/nm)

**Output**: one file per half-month with shape (224, 464) at NLDAS grid

In [ ]:
sif_files = sorted(sif_dir.glob('sif_ann_*.nc4'))

if not sif_files:
    print('WARNING: No SIF .nc4 files found in', sif_dir)
else:
    print(f'Found {len(sif_files)} SIF half-monthly files')

LON_MIN, LON_MAX = CONUS_LON[0] - 0.0625, CONUS_LON[-1] + 0.0625
LAT_MIN, LAT_MAX = CONUS_LAT[-1] - 0.0625, CONUS_LAT[0] + 0.0625

processed_sif = []
skipped_sif   = []
missing_sif   = []

for fpath in sif_files:
    stem = fpath.stem

    try:
        date_part = stem.replace('sif_ann_', '')
        yyyymm    = date_part[:6]
        half      = date_part[6] if len(date_part) > 6 and date_part[6] in ('a', 'b') else 'a'
        year      = int(yyyymm[:4])
    except (IndexError, ValueError):
        print(f'WARNING: Could not parse date from: {fpath.name}')
        skipped_sif.append(fpath.name)
        continue

    if year not in YEARS:
        continue

    out_name = 'SIF_CONUS_' + yyyymm + half + '.nc'
    dst_path = proc_dir / 'sif' / out_name

    if dst_path.exists() and not needs_reprocess(dst_path):
        processed_sif.append(out_name)
        continue

    if dst_path.exists():
        dst_path.unlink()

    try:
        ds = xr.open_dataset(fpath)
    except Exception as e:
        print(f'WARNING: Could not open {fpath.name}: {e}')
        missing_sif.append(fpath.name)
        continue

    sif_var = None
    for candidate in ['sif_ann', 'SIF_ann', 'SIF', 'sif']:
        if candidate in ds.data_vars:
            sif_var = candidate
            break
    if sif_var is None:
        print(f'WARNING: sif_ann not found in {fpath.name} | vars: {list(ds.data_vars)}')
        ds.close()
        missing_sif.append(fpath.name)
        continue

    da = ds[sif_var].squeeze(drop=True)

    lat_dim = lon_dim = None
    for dim in da.dims:
        dl = dim.lower()
        if dl in ('latitude', 'lat', 'y'): lat_dim = dim
        if dl in ('longitude', 'lon', 'x'): lon_dim = dim

    if lat_dim is None or lon_dim is None:
        print(f'WARNING: Could not identify lat/lon dims in {fpath.name}')
        ds.close()
        missing_sif.append(fpath.name)
        continue

    # Subset to clipped extent before regridding
    lat_vals = da[lat_dim].values
    lon_vals = da[lon_dim].values
    lat_mask = (lat_vals >= LAT_MIN) & (lat_vals <= LAT_MAX)
    lon_mask = (lon_vals >= LON_MIN) & (lon_vals <= LON_MAX)
    da = da.isel({lat_dim: lat_mask, lon_dim: lon_mask})

    rename_map = {}
    if lat_dim != 'lat': rename_map[lat_dim] = 'lat'
    if lon_dim != 'lon': rename_map[lon_dim] = 'lon'
    if rename_map:
        da = da.rename(rename_map)

    # Interpolate to clipped CONUS grid
    try:
        da_regrid = da.interp(
            lat=xr.DataArray(CONUS_LAT, dims='lat'),
            lon=xr.DataArray(CONUS_LON, dims='lon'),
            method='nearest',
            kwargs={'fill_value': np.nan},
        )
    except Exception as e:
        print(f'WARNING: interp failed for {fpath.name}: {e} — using reindex')
        da_regrid = da.reindex(lat=CONUS_LAT, lon=CONUS_LON, method='nearest', tolerance=0.07)

    ds_out = da_regrid.to_dataset(name='sif_ann')
    ds_out['sif_ann'].attrs.update({
        'units'      : 'mW/m^2/sr/nm',
        'long_name'  : 'Solar-Induced Chlorophyll Fluorescence (annualized)',
        'source'     : 'OCO-2 v11r',
        'yyyymm'     : yyyymm,
        'half_month' : half,
    })
    ds_out.attrs['crs'] = CONUS_CRS
    ds_out.to_netcdf(dst_path)
    ds.close()

    processed_sif.append(out_name)

print(f'\nSIF summary: processed {len(processed_sif)} half-month files | '
      f'missing/skipped {len(missing_sif) + len(skipped_sif)}')

# QC dims
qc_sif = proc_dir / 'sif' / 'SIF_CONUS_202207a.nc'
if qc_sif.exists():
    ds_chk = xr.open_dataset(qc_sif)
    lat_n = len(ds_chk.lat) if 'lat' in ds_chk.coords else '?'
    lon_n = len(ds_chk.lon) if 'lon' in ds_chk.coords else '?'
    print(f'QC dims: {lon_n} lon × {lat_n} lat  (expected {CONUS_WIDTH}×{CONUS_HEIGHT})')
    ds_chk.close()

expected_sif  = [str(y) + str(m).zfill(2) + h for y in YEARS for m in range(1, 13) for h in ('a', 'b')]
available_sif = {f.replace('SIF_CONUS_', '').replace('.nc', '') for f in processed_sif}
missing_sif_months = [ym for ym in expected_sif if ym not in available_sif]
if missing_sif_months:
    print(f'WARNING: Missing SIF half-months ({len(missing_sif_months)}): {missing_sif_months[:10]}')
else:
    print('All 240 expected SIF half-month files present.')

Found 251 SIF half-monthly files

SIF summary: processed 235 half-month files | missing/skipped 0
QC dims: 325 lon × 189 lat  (expected 325×189)


## 6. USDM — Drought Monitor Categories

Loads USDM bimonthly CONUS rasters (from `19_download_drought_conus.py`) and verifies alignment with NLDAS grid. DM categories: −1=No drought, 0=D0 (Abnormally Dry), 1=D1, 2=D2, 3=D3, 4=D4 (Exceptional).

In [ ]:
usdm_files = sorted(usdm_dir.glob('USDM_CONUS_*.tif'))

if not usdm_files:
    print('WARNING: No USDM CONUS files found in', usdm_dir)
else:
    print(f'Found {len(usdm_files)} USDM CONUS files')

valid_usdm   = []
invalid_usdm = []

for fpath in usdm_files:
    # Skip monthly-aggregated files in the source dir (shouldn't exist, but guard)
    if 'monthly' in fpath.name:
        continue

    dst = proc_dir / 'drought_usdm' / fpath.name

    if dst.exists() and not needs_reprocess(dst):
        valid_usdm.append(fpath.name)
        continue

    if dst.exists():
        dst.unlink()

    try:
        with rasterio.open(fpath) as src:
            if src.height != FULL_HEIGHT or src.width != FULL_WIDTH:
                print(f'WARNING: unexpected source shape in {fpath.name}: {src.height}×{src.width}')
                invalid_usdm.append(fpath.name)
                continue
    except Exception as e:
        print(f'WARNING: Could not open {fpath.name}: {e}')
        invalid_usdm.append(fpath.name)
        continue

    crop_raster_to_openet(fpath, dst)
    valid_usdm.append(fpath.name)

print(f'Valid and cropped to {CONUS_WIDTH}×{CONUS_HEIGHT}: {len(valid_usdm)} | Invalid: {len(invalid_usdm)}')

# Build monthly mean from the two half-month files
usdm_monthly_created = []

for year in YEARS:
    for month in range(1, 13):
        yyyymm = str(year) + str(month).zfill(2)
        fa = proc_dir / 'drought_usdm' / ('USDM_CONUS_' + yyyymm + 'a.tif')
        fb = proc_dir / 'drought_usdm' / ('USDM_CONUS_' + yyyymm + 'b.tif')
        dst_monthly = proc_dir / 'drought_usdm' / ('USDM_CONUS_monthly_' + yyyymm + '.tif')

        if dst_monthly.exists() and not needs_reprocess(dst_monthly):
            usdm_monthly_created.append(yyyymm)
            continue

        if dst_monthly.exists():
            dst_monthly.unlink()

        have_a, have_b = fa.exists(), fb.exists()
        if not have_a and not have_b:
            continue

        if have_a and have_b:
            with rasterio.open(fa) as src_a:
                data_a = src_a.read(1).astype(np.float32)
                meta   = src_a.meta.copy()
            with rasterio.open(fb) as src_b:
                data_b = src_b.read(1).astype(np.float32)
            nodata = meta.get('nodata', -9999)
            if nodata is not None:
                data_a[data_a == nodata] = np.nan
                data_b[data_b == nodata] = np.nan
            monthly_mean = np.nanmean(np.stack([data_a, data_b], axis=0), axis=0)
        else:
            src_file = fa if have_a else fb
            with rasterio.open(src_file) as src_a:
                monthly_mean = src_a.read(1).astype(np.float32)
                meta         = src_a.meta.copy()

        meta.update({'dtype': 'float32', 'nodata': np.nan, 'driver': 'GTiff'})
        with rasterio.open(dst_monthly, 'w', **meta) as dst:
            dst.write(monthly_mean, 1)
        usdm_monthly_created.append(yyyymm)

print(f'Monthly USDM files created/present: {len(usdm_monthly_created)}')

# Sample QC
sample_usdm = proc_dir / 'drought_usdm' / 'USDM_CONUS_monthly_202207.tif'
if sample_usdm.exists():
    with rasterio.open(sample_usdm) as src:
        data = src.read(1).astype(np.float32)
        if src.nodata is not None:
            data[data == src.nodata] = np.nan
        valid  = data[np.isfinite(data)]
        frac_d = float(np.sum(valid > 0) / len(valid))
        print(f'\nSample — USDM July 2022 ({src.width}×{src.height}, CRS: {src.crs.to_epsg()}):')
        print(f'  Fraction with any drought (DM>0): {frac_d*100:.1f}%')

Found 240 USDM CONUS files
Valid and cropped to 325×189: 240 | Invalid: 0
Monthly USDM files created/present: 0


## 7. GRIDMET — Drought Indices (SPI, SPEI, EDDI, PDSI)

Loads multi-band GRIDMET drought GeoTIFFs and verifies alignment. Already on NLDAS grid from `19_download_drought_conus.py`.

**Bands**: `spi30d`, `spi90d`, `spei30d`, `spei90d`, `eddi30d`, `eddi90d`, `pdsi`, `z`

**Drought thresholds used in analysis**:
- SPI/SPEI: < −2.0 = D4, −2 to −1.5 = D3, −1.5 to −1.0 = D2, −1.0 to −0.5 = D1, −0.5 to 0 = D0, > 0 = No drought
- EDDI: > 2.0 = D4, 1.5–2.0 = D3, 1.0–1.5 = D2, 0.5–1.0 = D1, 0–0.5 = D0, < 0 = No drought
- PDSI: < −4 = D4, −4 to −3 = D3, −3 to −2 = D2, −2 to −1 = D1, −1 to 1 = No drought

In [ ]:
GRIDMET_BAND_NAMES = ['spi30d', 'spi90d', 'spei30d', 'spei90d', 'eddi30d', 'eddi90d', 'pdsi', 'z']

gmet_files = sorted(gmet_dir.glob('GRIDMET_drought_*.tif'))

if not gmet_files:
    print('WARNING: No GRIDMET drought files found in', gmet_dir)
else:
    print(f'Found {len(gmet_files)} GRIDMET drought files')

valid_gmet   = []
invalid_gmet = []
sample_gmet  = None

for fpath in gmet_files:
    dst = proc_dir / 'drought_gridmet' / fpath.name

    if dst.exists() and not needs_reprocess(dst):
        valid_gmet.append(fpath.name)
        if sample_gmet is None:
            sample_gmet = dst
        continue

    if dst.exists():
        dst.unlink()

    try:
        with rasterio.open(fpath) as src:
            if src.height != FULL_HEIGHT or src.width != FULL_WIDTH:
                print(f'WARNING: unexpected source shape in {fpath.name}: {src.height}×{src.width}')
                invalid_gmet.append(fpath.name)
                continue
    except Exception as e:
        print(f'WARNING: Could not open {fpath.name}: {e}')
        invalid_gmet.append(fpath.name)
        continue

    crop_raster_to_openet(fpath, dst)
    valid_gmet.append(fpath.name)
    if sample_gmet is None:
        sample_gmet = dst

print(f'Valid and cropped to {CONUS_WIDTH}×{CONUS_HEIGHT}: {len(valid_gmet)} | Invalid: {len(invalid_gmet)}')

if sample_gmet and sample_gmet.exists():
    with rasterio.open(sample_gmet) as src:
        print(f'\nSample: {sample_gmet.name} | {src.count} bands | {src.width}×{src.height} | CRS: {src.crs.to_epsg()}')
        for b in range(1, src.count + 1):
            arr   = src.read(b, masked=True)
            label = GRIDMET_BAND_NAMES[b - 1] if b - 1 < len(GRIDMET_BAND_NAMES) else f'band{b}'
            print(f'  band {b} ({label}): min={float(arr.min()):.3f}  max={float(arr.max()):.3f}  mean={float(arr.mean()):.3f}')

expected_months = [str(y) + str(m).zfill(2) for y in YEARS for m in range(1, 13)]
available_stems = {f.replace('GRIDMET_drought_', '').replace('.tif', '') for f in valid_gmet}
missing_months  = [ym for ym in expected_months if ym not in available_stems]
if missing_months:
    print(f'\nWARNING: Missing GRIDMET months ({len(missing_months)}): {missing_months[:12]}')
else:
    print('All 120 expected GRIDMET drought months present.')

Found 120 GRIDMET drought files
Valid and cropped to 325×189: 120 | Invalid: 0

Sample: GRIDMET_drought_201501.tif | 8 bands | 325×189 | CRS: 4326
  band 1 (spi30d): min=-1.581  max=2.090  mean=0.160
  band 2 (spi90d): min=-2.090  max=2.090  mean=-0.037
  band 3 (spei30d): min=-1.564  max=1.911  mean=0.181
  band 4 (spei90d): min=-1.787  max=2.090  mean=-0.046
  band 5 (eddi30d): min=-2.057  max=2.013  mean=-0.159
  band 6 (eddi90d): min=-2.090  max=2.090  mean=0.032
  band 7 (pdsi): min=-6.373  max=6.306  mean=-0.111
  band 8 (z): min=-3.203  max=4.027  mean=0.023
All 120 expected GRIDMET drought months present.


## 8. Coverage Summary

Check how many months of each variable are available in the processed CONUS directory.

In [ ]:
# Count files per variable sub-directory
def count_proc_files(subdir, pattern, expected):
    """Count processed files matching a pattern and return (available, missing_count, missing_list)."""
    d = proc_dir / subdir
    if not d.exists():
        return 0, expected, []
    found = list(d.glob(pattern))
    n = len(found)
    return n, expected - n, found

specs = [
    # (label, subdir, glob_pattern, expected_count)
    ('CDL',              'cdl',             'CDL_CONUS_*.tif',              10),
    ('OpenET',           'openet',          'OpenET_CONUS_*.tif',          120),
    ('NLDAS',            'nldas',           'NLDAS_Evap_*.nc',             120),
    ('NDVI',             'ndvi',            'NDVI_CONUS_*.tif',            120),
    ('SIF',              'sif',             'SIF_CONUS_*.nc',              240),
    ('USDM (bimonthly)', 'drought_usdm',    'USDM_CONUS_[0-9]*.tif',      240),
    ('GRIDMET drought',  'drought_gridmet', 'GRIDMET_drought_*.tif',       120),
]

print('Variable         | Expected | Available | Missing')
print('-' * 52)

detail_missing = {}

for label, subdir, pat, expected in specs:
    n, miss_count, found = count_proc_files(subdir, pat, expected)
    # Clamp missing to 0 in case extra files exist
    miss_display = max(0, miss_count)
    print(
        label.ljust(17)
        + '|' + str(expected).rjust(9)
        + ' |' + str(n).rjust(9)
        + ' |' + str(miss_display).rjust(8)
    )
    detail_missing[label] = (n, expected, miss_display)

print('')
print('Details on missing files by variable:')

# For each variable, list specific missing months
def get_available_stems(subdir, pat, prefix, suffix):
    """Return set of YYYYMM stems from proc_dir/subdir files."""
    d = proc_dir / subdir
    if not d.exists():
        return set()
    return set(f.name.replace(prefix, '').replace(suffix, '') for f in d.glob(pat))

# CDL: per year
cdl_avail = get_available_stems('cdl', 'CDL_CONUS_*.tif', 'CDL_CONUS_', '.tif')
cdl_missing = [str(y) for y in YEARS if str(y) not in cdl_avail]
if cdl_missing:
    print('  CDL missing years:', cdl_missing)
else:
    print('  CDL: all years present')

# OpenET: per YYYYMM
openet_avail = get_available_stems('openet', 'OpenET_CONUS_*.tif', 'OpenET_CONUS_', '.tif')
openet_exp   = [str(y) + str(m).zfill(2) for y in YEARS for m in range(1, 13)]
openet_miss  = [ym for ym in openet_exp if ym not in openet_avail]
if openet_miss:
    print('  OpenET missing months (' + str(len(openet_miss)) + '):', openet_miss[:6],
          '...' if len(openet_miss) > 6 else '')
else:
    print('  OpenET: all months present')

# NLDAS: per YYYYMM
nldas_avail = get_available_stems('nldas', 'NLDAS_Evap_*.nc', 'NLDAS_Evap_', '.nc')
nldas_exp   = [str(y) + str(m).zfill(2) for y in YEARS for m in range(1, 13)]
nldas_miss  = [ym for ym in nldas_exp if ym not in nldas_avail]
if nldas_miss:
    print('  NLDAS missing months (' + str(len(nldas_miss)) + '):', nldas_miss[:6],
          '...' if len(nldas_miss) > 6 else '')
else:
    print('  NLDAS: all months present')

# NDVI: per YYYYMM
ndvi_avail = get_available_stems('ndvi', 'NDVI_CONUS_*.tif', 'NDVI_CONUS_', '.tif')
ndvi_exp   = [str(y) + str(m).zfill(2) for y in YEARS for m in range(1, 13)]
ndvi_miss  = [ym for ym in ndvi_exp if ym not in ndvi_avail]
if ndvi_miss:
    print('  NDVI missing months (' + str(len(ndvi_miss)) + '):', ndvi_miss[:6],
          '...' if len(ndvi_miss) > 6 else '')
else:
    print('  NDVI: all months present')

# SIF: per YYYYMMh
sif_avail = get_available_stems('sif', 'SIF_CONUS_*.nc', 'SIF_CONUS_', '.nc')
sif_exp   = [str(y) + str(m).zfill(2) + h for y in YEARS for m in range(1, 13) for h in ('a', 'b')]
sif_miss  = [ym for ym in sif_exp if ym not in sif_avail]
if sif_miss:
    print('  SIF missing half-months (' + str(len(sif_miss)) + '):', sif_miss[:6],
          '...' if len(sif_miss) > 6 else '')
else:
    print('  SIF: all half-months present')

# USDM: count bimonthly (a+b) files
usdm_d = proc_dir / 'drought_usdm'
usdm_bimonthly = list(usdm_d.glob('USDM_CONUS_[0-9]*.tif')) if usdm_d.exists() else []
# Only count non-monthly files (no 'monthly' in name)
usdm_bimonthly = [f for f in usdm_bimonthly if 'monthly' not in f.name]
print('  USDM bimonthly files present:', len(usdm_bimonthly), '(expected 240)')

# GRIDMET
gmet_avail = get_available_stems('drought_gridmet', 'GRIDMET_drought_*.tif', 'GRIDMET_drought_', '.tif')
gmet_exp   = [str(y) + str(m).zfill(2) for y in YEARS for m in range(1, 13)]
gmet_miss  = [ym for ym in gmet_exp if ym not in gmet_avail]
if gmet_miss:
    print('  GRIDMET missing months (' + str(len(gmet_miss)) + '):', gmet_miss[:6],
          '...' if len(gmet_miss) > 6 else '')
else:
    print('  GRIDMET: all months present')

Variable         | Expected | Available | Missing
----------------------------------------------------
CDL              |       10 |       10 |       0
OpenET           |      120 |      120 |       0
NLDAS            |      120 |      120 |       0
NDVI             |      120 |      120 |       0
SIF              |      240 |      235 |       5
USDM (bimonthly) |      240 |      240 |       0
GRIDMET drought  |      120 |      120 |       0

Details on missing files by variable:
  CDL: all years present
  OpenET: all months present
  NLDAS: all months present
  NDVI: all months present
  SIF missing half-months (5): ['201708a', '201708b', '201709a', '201801a', '201801b'] 
  USDM bimonthly files present: 240 (expected 240)
  GRIDMET: all months present


## 9. Quick QC Maps

Plot one map per variable for July 2022 (a moderate drought year) to verify spatial alignment and value ranges.

In [ ]:
import warnings

fig_dir = project_root / 'figures' / 'conus'
fig_dir.mkdir(parents=True, exist_ok=True)

def load_raster_band(fpath, band=1, nodata_to_nan=True):
    fpath = Path(fpath)
    if fpath.suffix in ('.nc', '.nc4'):
        ds  = xr.open_dataset(fpath)
        var = list(ds.data_vars)[0]
        arr = ds[var].squeeze().values.astype(np.float32)
        ds.close()
    else:
        with rasterio.open(fpath) as src:
            arr = src.read(band).astype(np.float32)
            if nodata_to_nan and src.nodata is not None:
                arr[arr == src.nodata] = np.nan
    return arr

panels = [
    ('CDL Cropland Frac.',    proc_dir / 'cdl'            / 'CDL_CONUS_2022.tif',                 'Greens',   0.0,  1.0, 1),
    ('OpenET (mm/mo)',        proc_dir / 'openet'          / 'OpenET_CONUS_202207.tif',             'Blues',    0,  150, 1),
    ('NLDAS Evap (mm/mo)',    proc_dir / 'nldas'           / 'NLDAS_Evap_202207.nc',               'Blues',    0,  150, 1),
    ('NDVI',                  proc_dir / 'ndvi'            / 'NDVI_CONUS_202207.tif',              'YlGn',  -0.1,  0.9, 1),
    ('SIF (mW/m²/sr/nm)',     proc_dir / 'sif'             / 'SIF_CONUS_202207a.nc',               'YlOrRd',   0,  3.0, 1),
    ('USDM Category',         proc_dir / 'drought_usdm'    / 'USDM_CONUS_monthly_202207.tif',      'RdYlGn_r',-1,  4,   1),
    ('SPEI-90d',              proc_dir / 'drought_gridmet' / 'GRIDMET_drought_202207.tif',         'BrBG',    -3,  3,   3),
]

fig, axes = plt.subplots(3, 3, figsize=(18, 10))
axes_flat = axes.flatten()

# Plot extent uses the clipped CONUS_LON/LAT
plot_extent = [CONUS_LON[0] - 0.0625, CONUS_LON[-1] + 0.0625,
               CONUS_LAT[-1] - 0.0625, CONUS_LAT[0] + 0.0625]

for idx, (title, fpath, cmap, vmin, vmax, band) in enumerate(panels):
    ax = axes_flat[idx]
    fpath = Path(fpath)

    if not fpath.exists():
        ax.set_title(title, fontsize=9)
        ax.text(0.5, 0.5, 'Not yet processed', ha='center', va='center',
                transform=ax.transAxes, fontsize=8, color='gray')
        ax.axis('off')
        continue

    try:
        arr        = load_raster_band(fpath, band=band)
        arr_masked = np.where(np.isfinite(arr), arr, np.nan)

        im = ax.imshow(arr_masked, cmap=cmap, vmin=vmin, vmax=vmax,
                       extent=plot_extent, aspect='auto', interpolation='nearest')
        fig.colorbar(im, ax=ax, fraction=0.03, pad=0.02)
        ax.set_title(f'{title}\n{arr.shape[1]}×{arr.shape[0]}', fontsize=8)
        ax.set_xlabel('Longitude', fontsize=7)
        ax.set_ylabel('Latitude',  fontsize=7)
        ax.tick_params(labelsize=6)
    except Exception as e:
        ax.set_title(title, fontsize=9)
        ax.text(0.5, 0.5, f'Error:\n{e}', ha='center', va='center',
                transform=ax.transAxes, fontsize=7, color='red')
        ax.axis('off')

for idx in range(len(panels), len(axes_flat)):
    axes_flat[idx].set_visible(False)

fig.suptitle('CONUS QC Maps — July 2022 | All datasets clipped to OpenET extent (325×189)',
             fontsize=12, y=1.01)
plt.tight_layout()

out_fig = fig_dir / 'qc_maps_july2022.png'
plt.savefig(out_fig, dpi=150, bbox_inches='tight')
plt.show()
print('Saved:', out_fig)

Saved: /home/pielab-sandbox-jcoldiron/SIF-Analysis/figures/conus/qc_maps_july2022.png


## Done

All harmonized data is now in `data/processed/conus/`. Proceed to `01_human_et_conus.ipynb` for the analysis.